In [ ]:
from sentence_transformers import SentenceTransformer, util
from web_data_retrival import Data_Retrival
from newspaper import Article
import requests
from bs4 import BeautifulSoup

import re
import unicodedata
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import faiss

import os
os.environ["TRANSFORMERS_VERBOSITY"] = "info"

In [ ]:


def get_page_content(results):
    page_content = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    for page in results:
        url = page.get("Link")
        if not url:
            continue

        try:
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status() 

            soup = BeautifulSoup(response.text, "html.parser")

            if soup.find("article"):
                main_content = soup.find("article")
            elif soup.find("main"):
                main_content = soup.find("main")
            else:
                main_content = soup 

            text = main_content.get_text(separator="\n").strip()
            title = soup.title.string if soup.title else "No Title"

            page_content.append({"title": title, "content": text})

        except requests.exceptions.RequestException as e:
            print(f"Failed to fetch {url}: {e}")
            continue

    return page_content

In [99]:
def clean(text):
    text = unicodedata.normalize("NFKC",text)
    text = re.sub(r"\n+","\n",text)
    text = re.sub(r"\s+"," ",text)
    text = text.strip()
    text = re.sub(r'^\s*[-*•]\s*', '', text, flags=re.MULTILINE)
    return text

In [100]:
def get_top_k_results(model, user_input, chunks,top_k):
    
    model_mini = SentenceTransformer(model)
    q_emb = model_mini.encode([user_input])
    ans_emb = model_mini.encode(chunks)

    top_k_results = util.semantic_search(q_emb,ans_emb,top_k)
    return top_k_results

In [101]:
def retrive_chunks(emb_model,index,user_input, chunks, k = 5):
    q_encode = emb_model.encode([user_input], convert_to_numpy=True)
    distance, indicies = index.search(q_encode, k)
    return [chunks[i] for i in indicies[0]]

In [102]:
def phi_model_load(phi_model,phi_tokenizer,prompt,max_new_tokens = 300):
    inputs = phi_tokenizer(prompt,return_tensors="pt").to(phi_model.device)

    outputs = phi_model.generate(**inputs, max_new_tokens = 300, do_sample = False)

    return phi_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def answer_the_question(user_input):
    print(f"The question you entered is : {user_input}")
    obj = Data_Retrival(Query = user_input, max_results=7)
    results = obj.get_data()

    page_content = get_page_content(results)

    text = ""
    for page in page_content:
        print()
        text += page.get("title") + ": " + page.get("content") + "\n"
    text = clean(text)


    splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 90,length_function = len)
    chunks = splitter.split_text(text)

    top_k_results = get_top_k_results(model = "All-miniLM-L6-V2", user_input=user_input, chunks = chunks, top_k = 15)

    next_sum = ""
    for item in top_k_results[0]:
        next_sum += chunks[item["corpus_id"]]



    emb_model = SentenceTransformer("All-miniLM-L6-v2")

    ch_encode = emb_model.encode(chunks,convert_to_numpy=True)
    dimensions = ch_encode.shape[1]

    index = faiss.IndexFlatL2(dimensions)
    index.add(ch_encode)


    context_chunks = retrive_chunks(emb_model,index, user_input,chunks,5)
    context = "\n".join(context_chunks)


    device = "mps" if torch.backends.mps.is_available() else "cpu"

    print("Using Device: ",device)

    phi_model_name = "microsoft/phi-3-mini-4k-instruct"

    phi_tokenizer = AutoTokenizer.from_pretrained(phi_model_name)

    phi_model = AutoModelForCausalLM.from_pretrained(phi_model_name,dtype = torch.float16,device_map = "auto")


    phi_prompt = f"""<|system|>
    You are a medical assistant.
    Answer the following concisely in **TWO paragraphs** using only the context.
    Be concise and factual.
    </|system|>

    <|user|>
    Context:
    {context}

    Question:
    {user_input}

    </|user|>

    <|assistant|>"""

    string = phi_model_load(phi_model,phi_tokenizer,phi_prompt)

    s = string.split("</|user|>")[-1]

    final_answer = re.split(r"\n#|\n##|\n###",s)[0].strip()
    final_answer.split("\n\n\n")
    return final_answer[0] +"\n\n" + final_answer[1]



    

Enter Your Question : 


In [15]:
prompt = f"""Answer the question using ONLY the context below.

context:
{context}

Question: 
{user_input}

Answer concisely and medically accurate:
"""

In [ ]:

device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Using Device: ",device)

phi_model_name = "microsoft/phi-3-mini-4k-instruct"

phi_tokenizer = AutoTokenizer.from_pretrained(phi_model_name)

phi_model = AutoModelForCausalLM.from_pretrained(phi_model_name,dtype = torch.float16,device_map = "auto")


phi_prompt = f"""<|system|>
You are a medical assistant.
Answer the following concisely in **TWO paragraphs** using only the context.
Be concise and factual.
</|system|>

<|user|>
Context:
{context}

Question:
{user_input}

</|user|>

<|assistant|>"""

string = phi_model_load(phi_prompt)

s = string.split("</|user|>")[-1]

final_answer = re.split(r"\n#|\n##|\n###",s)[0].strip()
final_answer.split("\n\n\n")

Using Device:  mps


In [27]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_model_name = "google/flan-t5-large"
t5_tokenizer = T5Tokenizer.from_pretrained(t5_model_name)
t5_model = T5ForConditionalGeneration.from_pretrained(t5_model_name,dtype = torch.float16).to(device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [28]:
def T5_model_load(prmpt, max_new_tokens=200):
    inputs = t5_tokenizer(prompt, return_tensors = "pt",truncation = True).to(device)
    outputs = t5_model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample = False)

    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [75]:
def answer_question(prompt, model_type="phi"):
    if model_type == "phi":
        return phi_model_load(prompt)
    elif model_type == "t5":
        return T5_model_load(prompt)
    else:
        raise ValueError("model_type must be 'phi' or 't5'")

In [76]:
# answer_question(prompt,"phi")

In [77]:
# answer_question(prompt,"t5")

In [ ]:
phi_prompt = f"""<|system|>
You are a medical assistant.
Answer the following concisely in **TWO paragraphs** using only the context.
Be concise and factual.
</|system|>

<|user|>
Context:
{context}

Question:
{user_input}

</|user|>

<|assistant|>"""

string = answer_question(phi_prompt,"phi")

s = string.split("</|user|>")[-1]

final_answer = re.split(r"\n#|\n##|\n###",s)[0].strip()
final_answer.split("\n\n\n")

In [ ]:
!pip3 install 